# DATASET ANALYSIS

## Loading dataset

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('./results.csv')
df.head(10)

,song_name,genre,duration [s],sample_rate,processing_time_demucs,bpm_song,bpm_drums,bpm_hpss,key_song,confidence_song,key_demucs,confidence_demucs,key_hpss,confidence_hpss,manual_bpm,manual_key,agreement_bpm,agreement_key
0,Eric Clapton - Tears In Heaven.wav,acoustic,274.95,44100,154.23,152.00,152.00,152.00,A Major,0.6694,A Major,0.7211,A Major,0.6648,77,A Major,0.00,3
1,Extreme - More Than Words.wav,acoustic,255.56,44100,137.62,92.29,46.14,92.29,F# Major,0.6938,F# Major,0.3700,F# Major,0.6802,92,F# Major,21.75,3
2,Oasis - Wonderwall.wav,acoustic,279.50,44100,149.61,117.45,117.45,117.45,E Major,0.7251,E Major,0.6343,E Major,0.7387,87,F# Minor,0.00,3
3,The Beatles - Blackbird.wav,acoustic,138.41,44100,82.67,93.96,93.96,93.96,G Minor,0.3990,G Minor,0.3579,G Minor,0.3962,94,G Major,0.00,3
4,Tracy Chapman - Fast Car.wav,acoustic,266.26,44100,178.10,103.36,103.36,103.36,A Minor,0.4734,A Major,0.3703,A Minor,0.4777,102,A Major,0.00,2
5,Albert King - Born Under A Bad Sign.wav,blues,168.16,44100,126.21,92.29,92.29,92.29,C# Major,0.5091,G# Minor,0.6147,C# Major,0.5255,103,C# Minor,0.00,2
6,B.B. King - The Thrill Is Gone.wav,blues,329.07,44100,253.48,90.67,90.67,90.67,B Major,0.6200,B Minor,0.7077,B Minor,0.6250,91,B Minor,0.00,2
7,Robert Johnson - Cross Road Blues.wav,blues,157.97,44100,126.66,101.33,123.05,101.33,A# Minor,0.3822,B Minor,0.3493,B Minor,0.3774,101,B Major,10.24,2
8,Stevie Ray Vaughan Double Trouble - Pride and ...,blues,223.19,44100,172.84,126.05,126.05,126.05,D# Major,0.5541,D# Major,0.4976,D# Major,0.5536,126,D# Major,0.00,3
9,The Blues Brothers - Sweet Home Chicago.wav,blues,467.67,44100,367.61,126.05,126.05,126.05,E Major,0.8078,E Major,0.8640,E Major,0.8051,126,E Major,0.00,3


## BPM Error (ne uzimajući u obzir Half-time i Double-time)

In [6]:
def bpm_error(predicted, original):
    candidates = [predicted, predicted * 2, predicted / 2]
    errors = [abs(c - original) for c in candidates]

    return min(errors)

In [7]:
df['error_song'] = df.apply(lambda row: bpm_error(row['bpm_song'], row['manual_bpm']), axis=1)
df['error_drums'] = df.apply(lambda row: bpm_error(row['bpm_drums'], row['manual_bpm']), axis=1)
df['error_hpss'] = df.apply(lambda row: bpm_error(row['bpm_hpss'], row['manual_bpm']), axis=1)

## BPM MAE

In [10]:
mae_song = df['error_song'].mean()
mae_drums = df['error_drums'].mean()
mae_hpss = df['error_hpss'].mean()

print(f'Song MAE: {round(mae_song, 2)}')
print(f'Drums MAE: {round(mae_drums, 2)}')
print(f'HPSS MAE: {round(mae_hpss, 2)}')

Song MAE: 5.71
Drums MAE: 4.35
HPSS MAE: 4.48


## BPM Accuracy

In [11]:
TOLERANCE = 3

In [12]:
df["song_bpm_correct"] = (df["error_song"] <= TOLERANCE)
df["drums_bpm_correct"] = (df["error_drums"] <= TOLERANCE)
df["hpss_bpm_correct"] = (df["error_hpss"] <= TOLERANCE)

In [15]:
song_bpm_acc = (df["song_bpm_correct"].mean() * 100)
drums_bpm_acc = (df["drums_bpm_correct"].mean() * 100)
hpss_bpm_acc = (df["hpss_bpm_correct"].mean() * 100)

print(f"[BPM]: Song Accuracy:  {round(song_bpm_acc, 2)}%")
print(f"[BPM]: Drums Accuracy: {round(drums_bpm_acc, 2)}%")
print(f"[BPM]: HPSS Accuracy:  {round(hpss_bpm_acc, 2)}%")

[BPM]: Song Accuracy:  79.09%
[BPM]: Drums Accuracy: 82.73%
[BPM]: HPSS Accuracy:  84.55%


## Key Accuracy

In [16]:
df["song_key_correct"] = (df["key_song"]==df["manual_key"])
df["demucs_key_correct"] = (df["key_demucs"]==df["manual_key"])
df["hpss_key_correct"] = (df["key_hpss"]==df["manual_key"])

song_key_acc = (df["song_key_correct"].mean() * 100)
demucs_key_acc = (df["demucs_key_correct"].mean() * 100)
hpss_key_acc = (df["hpss_key_correct"].mean() * 100)

print(f"Song Key Accuracy:   {round(song_key_acc, 2)}%")
print(f"Demucs Accuracy:     {round(demucs_key_acc, 2)}%")
print(f"HPSS Accuracy:       {round(hpss_key_acc, 2)}%")

Song Key Accuracy:   47.27%
Demucs Accuracy:     53.64%
HPSS Accuracy:       47.27%


## Confidence analiza

In [17]:
bins = [0, 0.4, 0.6, 0.8, 1.0]

labels = [
    "0-0.4",
    "0.4-0.6",
    "0.6-0.8",
    "0.8-1.0"
]

In [18]:
df["conf_group"] = pd.cut(df["confidence_demucs"], bins=bins, labels=labels)

In [22]:
confidence_analysis = (df.groupby("conf_group")["demucs_key_correct"].mean() * 100)
print(confidence_analysis)

conf_group
0-0.4      46.153846
0.4-0.6    50.000000
0.6-0.8    55.357143
0.8-1.0    60.000000
Name: demucs_key_correct, dtype: float64


## Key Accuracy po žanru

In [26]:
genre_key = (
    df.groupby("genre")["demucs_key_correct"]
    .mean()
    * 100
).sort_values(ascending=False)

print(genre_key)

genre
folk          100.0
disco          80.0
electronic     80.0
reggae         80.0
rap            80.0
acoustic       60.0
country        60.0
blues          60.0
hiphop         60.0
kpop           60.0
rock           60.0
techno         60.0
metal          60.0
trance         60.0
house          40.0
rnb            40.0
pop            40.0
latin          40.0
punk           20.0
funk           20.0
soul           20.0
jazz            0.0
Name: demucs_key_correct, dtype: float64


In [27]:
summary = pd.DataFrame({

    "Method": [
        "Song",
        "Demucs",
        "HPSS"
    ],

    "BPM MAE": [
        mae_song,
        mae_drums,
        mae_hpss
    ],

    "BPM Accuracy": [
        song_bpm_acc,
        drums_bpm_acc,
        hpss_bpm_acc
    ],

    "Key Accuracy": [
        song_key_acc,
        demucs_key_acc,
        hpss_key_acc
    ]
})

summary

,Method,BPM MAE,BPM Accuracy,Key Accuracy
0,Song,5.711682,79.090909,47.272727
1,Demucs,4.348727,82.727273,53.636364
2,HPSS,4.479955,84.545455,47.272727
